# Job Finder — RAG Agent Development & Deployment

**ISA632 Group Project | Phase 2 of 2**

This notebook covers the second half of the RAG pipeline: enabling MLflow tracing, iterative prompt engineering, building and testing the LangChain agent, registering the model to Unity Catalog, and deploying it via the Databricks Agent Framework with a Review App.

**Run order:**
1. `01_DataPreparation.ipynb` — ingest documents, build Delta table, sync Vector Search index
2. `02_AgentDevelopment.ipynb` ← you are here

**Prerequisites:**
- `01_DataPreparation.ipynb` has been run and the Vector Search index `isa632_7474656346303369.boopatt.jobindex` is synced
- `agent.py` is present at `/Workspace/Users/boopatt@miamioh.edu/Project/agent.py`
- Catalog: `isa632_7474656346303369` | Schema: `boopatt`

## Step 1 — Install Dependencies

Install all required packages for the agent and MLflow integration. The kernel restarts automatically — re-run subsequent cells after the restart.

In [0]:
%pip install -qqq -U databricks-sdk databricks-langchain databricks-vectorsearch langsmith>=0.1.125 langchain==0.3.27 mlflow-skinny[databricks]>=3.4.0

dbutils.library.restartPython()

## Step 2 — Enable MLflow Tracing

Enable LangChain auto-logging so that every agent invocation is automatically traced and logged in MLflow. This captures the full chain of retrieval → prompt → LLM response, making it easy to inspect and debug the pipeline.

The `VS_INDEX_NAME` environment variable is set here and read by `agent.py` at runtime to locate the correct Vector Search index.

In [0]:
import os
import mlflow

mlflow.langchain.autolog()
mlflow.set_experiment("/Users/boopatt@miamioh.edu/JobFinder_AgentDev")

os.environ["VS_INDEX_NAME"] = "isa632_7474656346303369.boopatt.jobindex"
print("MLflow tracing enabled. VS_INDEX_NAME set.")

## Step 3 — Prompt Engineering: Before & After

A well-designed system prompt is critical for RAG agents. Our initial prompt produced three consistent failure modes that we diagnosed using Databricks Genie and fixed iteratively:

| Failure | Example | Root Cause |
|---|---|---|
| **Hallucination** | User asked "any available roles" → agent invented *"Senior Software Engineer Cloud Infrastructure"* as a search query | Vague input led the LLM to guess keywords not mentioned by the user |
| **Tool name mismatch** | Agent called `job_details_tool` which did not exist | `VectorSearchRetrieverTool` auto-generates its name from the index name (dots → double underscores); the prompt referenced a hardcoded name instead |
| **False matches** | User asked for entry-level roles → agent returned Manager/Senior roles without flagging the gap | No instruction to verify seniority match before presenting results |

The cells below show the original prompt and the improved version that addresses all three issues.

In [0]:
# Original system prompt — used in early testing
original_prompt = """
You are a job description expert and your task is to create job description summary
that are very good for anyone who asks about your available jobs.
Write a title and description that is similar to the following title and job description.
Maximum 300 words.
You must always call the {tool_name} tool to ground your answers.
Cite retrieved facts from search results in your response.
Return ONLY the final deliverable.
Do not describe your steps or tools. No code blocks, no brackets, no function names.

Format exactly:
Title: <one-line title>
Job Description: <~250 words>
"""

print("--- ORIGINAL PROMPT ---")
print(original_prompt)

**Problems with the original prompt:**
- `{tool_name}` was a static placeholder — it did not reflect the actual auto-generated tool name
- No instruction on *how* to query the tool (led to invented search terms)
- No rule to verify whether retrieved results actually match the user's request
- No fallback format when no matching results exist

The improved prompt below fixes all of these. The actual tool name is now derived dynamically from the index name (`vs_index_name.replace(".", "__")`) and injected via an f-string in `agent.py`.

In [0]:
# Improved system prompt — currently in agent.py
# tool_name is derived as vs_index_name.replace(".", "__") at runtime

improved_prompt = """
You are a job description assistant. Your ONLY data source is the {tool_name} retriever tool.

CRITICAL RULES:
1. ALWAYS call {tool_name} before responding. Never answer from your own knowledge.
2. When calling the tool, use the user's EXACT words or a close paraphrase as the query.
   Do NOT invent job titles, companies, or keywords the user did not mention.
   If the user's request is broad (e.g., "show me available roles"),
   use a general query like "available jobs" or "job openings".
3. Base your response ONLY on the documents returned by the tool.
   Do not add requirements, qualifications, or details not in the retrieved text.
4. AFTER retrieving results, verify whether they match the user's request:
   - Seniority level: Did the user want "entry level"/"junior" but you found "Manager"/"Senior"? → MISMATCH
   - Job function: Did the user want "marketing" but you found "engineering" roles? → MISMATCH
   - Any other explicit criteria the user mentioned (location, industry, etc.).
5. If there is a mismatch, do NOT present the results as if they satisfy the request.
   Use the MISMATCH FORMAT instead.

OUTPUT FORMAT when results MATCH the user's request:
Title: <exact or closely paraphrased title from the retrieved job posting>
Job Description: <250-word summary using only facts from retrieved documents.
                  Include role, key responsibilities, qualifications, and compensation if present.>

MISMATCH FORMAT when results do NOT match the user's request:
I was unable to find any [user's criteria, e.g., "entry-level"] positions in our current
job database. The available roles are at a different seniority level.

Here is what is currently available:
Title: <actual title from retrieved posting>
Level: <actual seniority level, e.g., Manager, Senior, Consultant>
Brief Summary: <2-3 sentence description of the role>

Would you like more details about any of these roles, or would you like to adjust your search criteria?
"""

print("--- IMPROVED PROMPT ---")
print(improved_prompt)

**Summary of changes:**

| # | New Rule | Problem It Solves |
|---|---|---|
| Rule 2 | Use user's exact words; default to generic terms for broad requests | Eliminates hallucinated search queries |
| Rule 3 | Only use facts from retrieved documents | Prevents LLM from adding unsourced details |
| Rule 4 | Explicitly verify seniority, job function, and other filters after retrieval | Catches mismatched results before they reach the user |
| Rule 5 | Prohibit presenting mismatched results as satisfying the request | Ensures honest, transparent responses |
| MISMATCH FORMAT | Dedicated output template for no-match scenarios | Gives users a useful fallback instead of a silent wrong answer |
| Tool name fix | Derive tool name dynamically via `vs_index_name.replace(".", "__")` | Fixes `tool not found` errors caused by auto-generated names |

## Step 4 — Load and Test the RAG Agent

Import the `AGENT` object from `agent.py`, which wraps the full LangChain pipeline with the improved prompt:
- **LLM:** Databricks Llama 4 Maverick via `ChatDatabricks`
- **Retriever:** Databricks Vector Search tool
- **Agent type:** Tool-calling agent via LangChain `AgentExecutor`

Run a test query with `verbose=True` to inspect the reasoning steps — you should see the agent call the vector search tool before generating a final answer.

In [0]:
import sys

# Ensure latest version of agent.py is loaded
sys.path.insert(0, '/Workspace/Users/boopatt@miamioh.edu/Project')
if 'agent' in sys.modules:
    del sys.modules['agent']

from agent import AGENT

AGENT.agent.verbose = True

user_input = "what are the available roles for scala?"
request = {
    "input": [
        {"role": "user", "content": user_input}
    ]
}

resp = AGENT.predict(request)
print(resp)

# Reset verbose before registration so serving logs stay clean
AGENT.agent.verbose = False

## Step 5 — Register the Model to Unity Catalog

Log the agent as an MLflow model and register it to Unity Catalog. Registering the model:
- Creates a versioned, shareable artifact in the catalog
- Captures all dependencies (LangChain, Vector Search, MLflow versions) for reproducibility
- Declares the Vector Search index as a resource so it is automatically provisioned at serving time

The registered model name is `isa632_7474656346303369.boopatt.getstarted_job_listings`.

In [0]:
import mlflow
from mlflow.models.resources import DatabricksVectorSearchIndex
from importlib.metadata import version

mlflow.set_registry_uri("databricks-uc")

model_name = "isa632_7474656346303369.boopatt.getstarted_job_listings"
vs_index_table_name = "isa632_7474656346303369.boopatt.jobindex"

mlflow.models.set_model(AGENT)

with mlflow.start_run(run_name="joblistings_demo") as run:
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            f"langchain=={version('langchain')}",
            f"databricks-vectorsearch=={version('databricks-vectorsearch')}",
            f"databricks_langchain=={version('databricks-langchain')}",
            f"mlflow=={version('mlflow')}"
        ],
        resources=[
            DatabricksVectorSearchIndex(index_name=vs_index_table_name)
        ]
    )

model_uri = f"runs:/{run.info.run_id}/{model_info.name}"
model_version = mlflow.register_model(model_uri, model_name)
registered_version = model_version.version
print(f"Model registered: {model_name} | Version: {registered_version}")

## Step 6 — Deploy with the Databricks Agent Framework

Deploy the registered model using `agents.deploy()`. This does two things automatically:
1. **Creates a Model Serving endpoint** — a scalable REST API for the chatbot
2. **Launches a Review App** — a built-in UI for testing and gathering human feedback on the model's responses

`scale_to_zero=True` keeps costs low by shutting the endpoint down when idle.

Deployment typically takes **15–20 minutes**. The cell will print a dot every 30 seconds while waiting.

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate
from databricks import agents

# Use registered_version captured in Step 5 — set manually if running this cell standalone
deploy_version = registered_version if 'registered_version' in dir() else 1

deployment_info = agents.deploy(
    model_name,
    model_version=deploy_version,
    scale_to_zero=True,
    environment_vars={"VS_INDEX_NAME": vs_index_table_name}
)

w = WorkspaceClient()
print(f"Deploying endpoint: {deployment_info.endpoint_name}")
print("Waiting for endpoint to be ready (15–20 min)...", end="", flush=True)

while (
    w.serving_endpoints.get(deployment_info.endpoint_name).state.ready == EndpointStateReady.NOT_READY
    or w.serving_endpoints.get(deployment_info.endpoint_name).state.config_update == EndpointStateConfigUpdate.IN_PROGRESS
):
    print(".", end="", flush=True)
    time.sleep(30)

print(f"\nEndpoint is ready: {deployment_info.endpoint_name}")
print(f"Review App URL: {deployment_info.review_app_url}")